In [1]:
import sys
import os
import numpy as np
import importlib

# Remontée de deux niveaux pour accéder à Data_loader
current_dir = os.getcwd()
project_root = os.path.normpath(os.path.join(current_dir, "..", ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Chemin du projet : {project_root}")

# Import du module de chargement des données
module_name = "RSA_deep_working.Data_loader.class_data_loaders"

try:
    class_data_loaders = importlib.import_module(module_name)
    DirectoryRSAClass = class_data_loaders.DirectoryRSAClass
except ModuleNotFoundError as e:
    print(f"Erreur lors de l'importation du module {module_name} : {e}")
    sys.exit(1)

Chemin du projet : /home/loai/Documents/code/RSMLExtraction


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import transforms
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import rsml
import tifffile


from RSA_deep_working.Data_loader.class_data_loaders import DirectoryRSAClass

# importing pretrained segmentation model
import segmentation_models_pytorch as smp
from torch.utils.tensorboard import SummaryWriter


2025-04-08 10:24:59.398651: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-04-08 10:24:59.526388: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1744100699.573885   10734 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1744100699.589588   10734 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1744100699.711139   10734 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

## Dataset and data loaders

In [3]:
base_directory = "/home/loai/Images/DataTest/UC1_data"

In [4]:
from RSA_deep_working.Data_loader.class_data_loaders import DirectoryRSAClass
import os
import numpy as np
import tifffile
import torch
from torch.utils.data import Dataset
import torchvision.transforms as transforms

# Dimensions d'origine et calculs
H, W = (1166, 1348)

H_new, W_new = (1184, 1376)
padding_bottom, padding_right = H_new - H, W_new - W # the opposite lol
print("pad :", padding_right, padding_bottom)

# Transformations pré-calculées
img_transform = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Resize((H_new, W_new), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.Pad(padding=(0, 0, padding_right, padding_bottom), fill=0),
    transforms.Normalize(mean=[0.5], std=[0.5])
])
mask_transform_series = transforms.Compose([
    transforms.ToTensor(),
    #transforms.Resize((H_new, W_new), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.Pad(padding=(0, 0, padding_right, padding_bottom), fill=0),
    
])
mask_transform_image = transforms.Compose([
    transforms.ToTensor(),
    
    #transforms.Resize((H_new, W_new), interpolation=transforms.InterpolationMode.NEAREST),
    transforms.Pad(padding=(0, 0, padding_right, padding_bottom), fill=0),
])

def mtg_transform(mtg):
    """
    Transforme le MTG en un tenseur PyTorch.
    """
    # Convertir le MTG en une représentation adaptée
    # Par exemple, convertir les coordonnées en tenseur
    # ou appliquer d'autres transformations spécifiques
    return mtg

pad : 28 18


In [ ]:
# Optimisation de la lecture des TIFF : mise en cache par fichier
class CachedTiffReader:
    def __init__(self):
        self.cache = {}

    def get_page(self, img_path, key):
        if img_path not in self.cache:
            # Chargement unique du fichier, stockage de toutes les pages
            with tifffile.TiffFile(img_path) as tif:
                self.cache[img_path] = [page.asarray() for page in tif.pages]
        return self.cache[img_path][key]

tiff_reader = CachedTiffReader()

class RSASeg2DDataset(Dataset):
    def __init__(self, rsa_dir_loader, mode='series', img_transform=None,
                 mask_transform_series=None, mask_transform_image=None, image_with_mtg=False, as_RGB=False):
        """
        mode: 'series' pour charger l'ensemble de la série temporelle,
              'image' pour charger image par image.
        """
        self.mode = mode
        self.samples = []  # contiendra les tuples en fonction du mode
        self.img_transform = img_transform
        self.mask_transform_series = mask_transform_series
        self.mask_transform_image = mask_transform_image
        self.image_with_mtg = image_with_mtg
        self.as_RGB = as_RGB

        for loader in rsa_dir_loader.loaders:
            img_path = loader.image_stack_path
            mask_path = loader.date_map_path
            mtg_path = loader.rsml_default_file if os.path.exists(loader.rsml_default_file) else loader.rsml_expert_file

            # Lecture du nombre de slices dans la série
            with tifffile.TiffFile(img_path) as tif:
                num_slices = len(tif.pages)

            if mode == 'series':
                # Une entrée par série temporelle complète, on stocke num_slices
                self.samples.append((img_path, mask_path, num_slices, mtg_path))
            elif mode == 'image':
                # Une entrée par image (slice)
                for z in range(num_slices):
                    self.samples.append((img_path, mask_path, z, mtg_path))
            else:
                raise ValueError("Mode non reconnu, choisissez 'series' ou 'image'")

    def __len__(self):
        return len(self.samples)
    
    def num_times(self, idx):
        """
        Retourne le nombre de slices (temps) que comporte la série.
        """
        img_path, _, _, _ = self.samples[idx]
        with tifffile.TiffFile(img_path) as tif:
            num_slices = len(tif.pages)
        return num_slices

    def __getitem__(self, idx):
        # Extraction des informations de l'échantillon
        img_path, mask_path, slice_info, mtg_path = self.samples[idx]

        if self.mode == 'series':
            # Mode série : chargement complet de la série d'images
            img = tifffile.imread(img_path)
            mask_raw = tifffile.imread(mask_path)
            extra_info = slice_info  # ici, slice_info correspond à num_slices

            if self.img_transform:
                img = self.img_transform(img)
            # Appliquer la transformation du masque ou utiliser le masque brut
            mask = self.mask_transform_series(mask_raw) if self.mask_transform_series else mask_raw

        else:  # mode 'image'
            # Mode image : slice_info correspond à l'index de la slice (z)
            z = slice_info
            img = tiff_reader.get_page(img_path, z)
            mask_raw = tifffile.imread(mask_path)
            extra_info = z

            # Calcul vectorisé du masque : les pixels non nuls et <= z valent 1, sinon 0
            mask = np.where((mask_raw != 0) & (mask_raw <= z + 1), 1, 0)

            if self.img_transform:
                img = self.img_transform(img)
            mask = self.mask_transform_image(mask) if self.mask_transform_image else mask

        if (self.as_RGB):
            img = img.repeat(3, 1, 1)
        # Gestion du retour : si image_with_mtg est activé, on retourne mtg_path, sinon un tensor nul
        additional = mtg_path if self.image_with_mtg else torch.tensor(0)
        #mask to float64 
        mask = mask.float()
        return img, mask, extra_info, additional


In [6]:

dir_loader = DirectoryRSAClass(base_directory, load_date_map=True, lazy=True)

# Pour entraîner image par image :
rsa_dataset_image = RSASeg2DDataset(
    dir_loader,
    mode='image',
    img_transform=img_transform,
    mask_transform_image=mask_transform_image,  # pipeline dédié
    image_with_mtg=True
)

print("Nombre d'échantillons :", len(rsa_dataset_image), "images\n")

# for reproducibility
torch.manual_seed(42)
np.random.seed(42)

generator = torch.Generator().manual_seed(42)

train_set, val_set, test_set = torch.utils.data.random_split(
    rsa_dataset_image,
    [int(len(rsa_dataset_image) * 0.7), int(len(rsa_dataset_image) * 0.2),
     int(len(rsa_dataset_image) * 0.1)+2],
    generator=generator
)

# Affichage des tailles des ensembles
print("Ensemble d'entraînement (image) :", len(train_set))
print("Ensemble de validation (image) :", len(val_set))
print("Ensemble de test (image) :", len(test_set))

Nombre d'échantillons : 754 images

Ensemble d'entraînement (image) : 527
Ensemble de validation (image) : 150
Ensemble de test (image) : 77


### 2D Image loaders

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Utilisation du device : {device}")

BATCH_SIZE = 2
# data loader optimization
train_loader = torch.utils.data.DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=8, pin_memory=True, worker_init_fn=lambda worker_id: np.random.seed(42 + worker_id))
val_loader = torch.utils.data.DataLoader(val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True, worker_init_fn=lambda worker_id: np.random.seed(42 + worker_id))
test_loader = torch.utils.data.DataLoader(test_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=8, pin_memory=True, worker_init_fn=lambda worker_id: np.random.seed(42 + worker_id))

Utilisation du device : cuda


## Model

In [8]:
model_BCE = smp.Unet(
    encoder_name="resnet34",
    encoder_depth=5,
    encoder_weights=None, # "imagenet",    
    in_channels=1,                 
    classes=1                      
)

model_Dice = smp.Unet(
    encoder_name="resnet34",       
    encoder_depth=5,
    encoder_weights=None, # "imagenet",    
    in_channels=1,                 
    classes=1                      
)

model_clDice = smp.Unet(
    encoder_name="resnet34",       
    encoder_depth=5,
    encoder_weights=None, # "imagenet",    
    in_channels=1,                 
    classes=1                      
)

model_Dice_clDice = smp.Unet(
    encoder_name="resnet34",       
    encoder_depth=5,
    encoder_weights=None, # "imagenet",    
    in_channels=1,                 
    classes=1                      
)

model_skRecall = smp.Unet(
    encoder_name="resnet34",       
    encoder_depth=5,
    encoder_weights=None, # "imagenet",    
    in_channels=1,                 
    classes=1                      
)

model_superVoxel = smp.Unet(
    encoder_name="resnet34",       
    encoder_depth=5,
    encoder_weights=None, # "imagenet",    
    in_channels=1,                 
    classes=1                      
)

model_skseg = smp.Unet(
    encoder_name="resnet34",       
    encoder_depth=5,
    encoder_weights=None, # "imagenet",    
    in_channels=1,                 
    classes=1                      
)

# Charger les poids pré-entraînés si disponibles
model_BCE.to(device)
model_Dice.to(device)
model_clDice.to(device)
model_skRecall.to(device)
model_superVoxel.to(device)

List_models = [model_BCE, model_Dice, model_clDice, model_Dice_clDice, model_skRecall, model_superVoxel, model_skseg]

## Evaluation

In [9]:
from PIL import Image

def tensor_to_heatmap_image(tensor, cmap='hot'):
    """
    Convertit un tenseur PyTorch en une heatmap image qui sera sauvagardée dans tensorboard.
    """
    # Normaliser le tenseur entre 0 et 1
    tensor = (tensor - tensor.min()) / (tensor.max() - tensor.min())
    
    cpu_tensor = tensor.cpu()
    
    # Convertir le tenseur en image PIL
    image = Image.fromarray((cpu_tensor.numpy() * 255).astype(np.uint8), mode='L')
    
    # Convertir l'image PIL en tableau numpy
    image_np = np.array(image)
    
    # Créer la heatmap
    heatmap = plt.get_cmap(cmap)(image_np)[:, :, :3]  # Ignorer l'alpha channel
    heatmap = (heatmap * 255).astype(np.uint8)  # Convertir en uint8
    
    return heatmap

In [10]:
import torch
import numpy as np
from tqdm import tqdm
import torchvision.transforms.functional as TF


def evaluate_segmentation(model, image, mask, mtg, metrics: list, prediction=None, threshold=0.5, writer=None, global_step=None, device='cpu'):
    """
    Évalue le modèle sur une image (ou un batch d'images) et calcule les métriques fournies.

    Args:
        model: Le modèle de segmentation.
        image: Image ou batch d'images.
        mask: Masque(s) correspondant(s).
        mtg: Informations supplémentaires (ex. métadonnées).
        metrics (list): Liste de fonctions de métriques à évaluer.
        prediction (tensor, optionnel): Prédiction pré-calculée.
        threshold (float): Seuil pour la binarisation.
        writer (SummaryWriter, optionnel): Writer TensorBoard pour loguer les métriques.
        global_step (int, optionnel): Pas global pour TensorBoard.
        device (str): Périphérique (cpu, cuda, etc.).

    Returns:
        dict: Dictionnaire contenant la prédiction et les scores calculés.
    """
    model.eval()
    image, mask = image.to(device), mask.to(device)

    # Calcul de la prédiction si non fournie
    if prediction is None:
        with torch.no_grad():
            output = model(image)
            prediction = (torch.sigmoid(output) > threshold).float()

    # Calcul des métriques via compréhension de dictionnaire
    scores = {metric.__name__: metric(
        prediction, mask, mtg=mtg) for metric in metrics}

    # Log des métriques dans TensorBoard si nécessaire
    if writer is not None and global_step is not None:
        for name, score in scores.items():
            writer.add_scalar(f"Eval/{name}", score, global_step)

    return {'prediction': prediction, 'scores': scores}


def evaluate_segmentation_on_loader(model, loader, metrics: list, threshold=0.5, writer=None, global_step=None, device='cpu'):
    """
    Évalue le modèle sur l'ensemble d'un DataLoader et accumule les scores pour chaque métrique.
    Enregistre également quelques exemples d'images, masques, prédictions et heatmaps dans TensorBoard.

    Args:
        model: Le modèle de segmentation.
        loader: DataLoader contenant les batches (images, masques, temps, mtgs).
        metrics (list): Liste de fonctions de métriques à évaluer.
        threshold (float): Seuil pour la binarisation.
        writer (SummaryWriter, optionnel): Writer TensorBoard pour loguer les métriques.
        global_step (int, optionnel): Pas global pour TensorBoard.
        device (str): Périphérique (cpu, cuda, etc.).

    Returns:
        dict: Dictionnaire contenant les scores finaux, ainsi que la concaténation des prédictions et masques.
    """
    model.eval()
    metric_scores = {metric.__name__: [] for metric in metrics}
    sample_batch = None  # Pour conserver un batch d'exemple pour l'affichage

    with torch.no_grad():
        for batch in tqdm(
            loader,
            desc="Evaluation iteration",
            postfix=f"Epoch: {global_step}" if global_step is not None else "",
            bar_format="{l_bar}{bar}{r_bar}",
            unit="batch",
            total=len(loader),
            position=0,
            leave=True,
            dynamic_ncols=True
        ):
            images, masks, time, mtgs = batch
            images, masks = images.to(device), masks.to(device)
            output = model(images)
            prediction = (torch.sigmoid(output) > threshold).float()

            # Calcul et accumulation des métriques
            for metric in metrics:
                score = metric(prediction, masks, time, mtg=mtgs[0])
                metric_scores[metric.__name__].append(score)

            # Enregistre le premier batch pour affichage dans TensorBoard
            if sample_batch is None and writer is not None and global_step is not None:
                sample_batch = (images.cpu(), masks.cpu(), output.cpu(), prediction.cpu())
            

    # Log des images si writer et global_step sont fournis et un batch exemple est disponible
    if writer is not None and global_step is not None:
        # for each metric, on calcule la moyenne
        for metric_name, scores in metric_scores.items():
            mean_score = np.mean(scores)
            writer.add_scalar(f"Eval/{metric_name}_mean", mean_score, global_step)
        
        # for each metric, on calcule la variance
        for metric_name, scores in metric_scores.items():
            var_score = np.var(scores)
            writer.add_scalar(f"Eval/{metric_name}_var", var_score, global_step)
    
    
        if sample_batch is not None:
            sample_images, sample_masks, sample_outputs, sample_preds = sample_batch
            n_samples = min(4, sample_images.shape[0])

            images = sample_images[:n_samples]
            masks = sample_masks[:n_samples]
            predictions = sample_preds[:n_samples]
            outputs = sample_outputs[:n_samples]

            sigmoid_heatmaps = []
            outputs_heatmaps = []
            for i in range(n_samples):
                # Supposons que l'output et la sigmoid sont des cartes 2D (dimension [1, H, W])
                out = outputs[i].squeeze()
                sig_out = torch.sigmoid(outputs[i]).squeeze()
                
                # Création de l'image heatmap
                out_img = tensor_to_heatmap_image(out, cmap='hot')
                sig_img = tensor_to_heatmap_image(sig_out, cmap='hot')
                
                # Conversion en tensor
                out_tensor = TF.to_tensor(out_img)
                sig_tensor = TF.to_tensor(sig_img)
                outputs_heatmaps.append(out_tensor)
                sigmoid_heatmaps.append(sig_tensor)
            
            outputs_heatmaps_tensor = torch.stack(outputs_heatmaps)
            sigmoid_heatmaps_tensor = torch.stack(sigmoid_heatmaps)
            
            # concaténation des images du sample
            images_concat = torch.cat([images[i] for i in range(n_samples)], dim=2)
            masks_concat = torch.cat([masks[i] for i in range(n_samples)], dim=2)
            predictions_concat = torch.cat([predictions[i] for i in range(n_samples)], dim=2)
            outputs_heatmaps_concat = torch.cat([outputs_heatmaps_tensor[i] for i in range(n_samples)], dim=2)
            sigmoid_heatmaps_concat = torch.cat([sigmoid_heatmaps_tensor[i] for i in range(n_samples)], dim=2)
            # Enregistrement des images concaténées dans TensorBoard
            writer.add_image("Sample/Images", images_concat, global_step)
            writer.add_image("Sample/Masks", masks_concat, global_step)  
            writer.add_image("Sample/Predictions", predictions_concat, global_step)
            writer.add_image("Sample/Heatmaps", outputs_heatmaps_concat, global_step)
            writer.add_image("Sample/Sigmoid_Heatmaps", sigmoid_heatmaps_concat, global_step)

In [11]:
import RSA_deep_working.Metrics.simple_metrics as sm
import RSA_deep_working.Metrics.topo_explicit_metrics as tm

metrics = sm.all_metrics() 
tubular_metrics = tm.all_metrics()    
all_metrics = []
for metric in metrics:
    all_metrics.append(metric)
for metric in tubular_metrics:
    all_metrics.append(metric)

## Training

In [12]:
# Train the model
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10, device='cpu', writer=None, scheduler=None, lr_change_threshold=1e-6):
    torch.cuda.empty_cache()
    model.train()
    
    for epoch in range(num_epochs):
        running_loss = 0.0
        for batch in tqdm(
                train_loader, 
                desc=f"Loss: {running_loss:.1e} / {len(train_loader)}, LR: {optimizer.param_groups[0]['lr']:.1e}", 
                bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [Elapsed: {elapsed} | Remaining: {remaining}]", 
                unit="batch", 
                total=len(train_loader), 
                position=0,
                postfix=f"Epoch: {epoch+1}/{num_epochs}",
                leave=True, 
                dynamic_ncols=True
            ):
            
            images, masks, _, _ = batch
            images = images.to(device)
            masks = masks.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, masks)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        epoch_loss = running_loss / len(train_loader)
        #print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {epoch_loss:.4f}")
        torch.cuda.empty_cache()

        # Log the loss to TensorBoard
        if writer is not None:
            writer.add_scalar("Loss/train", epoch_loss, epoch)
            writer.add_scalar("Learning Rate", optimizer.param_groups[0]['lr'], epoch)

        # Evaluation on validation set
        if val_loader is not None:
            evaluate_segmentation_on_loader(
                model,
                val_loader,
                metrics=all_metrics,
                threshold=0.5,
                writer=writer,
                global_step=epoch,
                device=device
            )
            
        # Mise à jour du scheduler et vérification de l'évolution de la loss
        if scheduler is not None:
            scheduler.step(epoch_loss)
        torch.cuda.empty_cache()

    return model

In [13]:
# cuda - clear cache
torch.cuda.empty_cache()

In [14]:
# adam optimizer with weight decay for each model
LR = 5e-5
def get_optimizer(model, lr=5e-5):
    return optim.Adam(model.parameters(), lr=lr)

optimizers = []
for model in List_models:
    optimizers.append(get_optimizer(model, lr=LR))

In [15]:
def cldice(prediction, mask, time=0, mtg=None):
    """
    Custom clDice loss function.
    """
    return 1 - tm.cldice(prediction, mask, time, mtg)

def bce(prediction, mask, time=0, mtg=None):
    """
    Binary Cross Entropy loss function.
    """
    return F.binary_cross_entropy_with_logits(prediction, mask)

def dice(prediction, mask, time=0, mtg=None):
    """
    Dice loss function.
    """
    return smp.losses.DiceLoss(mode='binary')(prediction, mask)

def dice_cldice(prediction, mask, time=0, mtg=None):
    """
    Combined Dice and clDice loss function.
    """
    return 0.5 * smp.losses.DiceLoss(mode='binary')(prediction, mask) + 0.5 * cldice(prediction, mask, time, mtg)

# list of loss functions corresponding to the models
loss_functions = [
    bce,
    dice,
    cldice,
    dice_cldice,
    tm.skeleton_recall,
    tm.Connectivity_Preserving_Instance_Segmentation,
    tm.evaluate_sk_seg,
]

In [16]:
# one writer for each model
writers = []
writer_names = [
    "BCE",
    "Dice",
    "clDice",
    "Dice_clDice",
    "skRecall",
    "superVoxel",
    "skseg"
]

for i in range(len(loss_functions)):
    writer = SummaryWriter(f"runs/uc1_segmentation_{writer_names[i]}")
    writers.append(writer)
global_steps = [0] * len(loss_functions)

In [17]:
def train_and_evaluate(model, loss_function, optimizer, writer, writer_name, train_loader=train_loader, val_loader=val_loader, test_loader=test_loader, metrics=all_metrics, num_epochs=50, scheduler=None, lr_change_threshold=1e-8, device='cpu'):
    # TensorBoard writer
    writer = SummaryWriter(log_dir=f"runs/uc1_segmentation_{writer_name}")
    
    # Training
    model = train_model(
        model,
        train_loader,
        val_loader,
        loss_function,
        optimizer,
        num_epochs=num_epochs,
        device=device,
        writer=writer
    )

    # Evaluation on validation set
    evaluate_segmentation_on_loader(
        model,
        val_loader,
        metrics=metrics,
        threshold=0.5,
        writer=writer,
        global_step=num_epochs,
        device=device
    )

    # Close the TensorBoard writer
    writer.close()
    # save the model
    torch.save(model.state_dict(), f"Unet_{writer_name}.pth")
    # return the model
    return model

In [18]:
# for each model, train and evaluate
i = 0
for model, loss_function in zip(List_models, loss_functions):
    # Création du scheduler pour un optimizer avec une patience de 5 époques et une réduction par un facteur 0.5
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizers[i], mode='min', patience=5, factor=LR * .5, threshold=1-8)
    
    trained_model = train_and_evaluate(
        model,
        loss_function,
        optimizers[i],
        writers[i],
        writer_name=writer_names[i],
        num_epochs=3,
        scheduler=scheduler,
        lr_change_threshold=1e-8,
        device=device,
        train_loader=train_loader,
        val_loader=val_loader
    )
    # Clear the cache
    torch.cuda.empty_cache()
    # empty memory 
    import gc
    gc.collect()
    i += 1

Loss: 0.0e+00 / 264, LR: 5.0e-05:  15%|█▍        | 39/264 [Elapsed: 00:23 | Remaining: 02:17]


KeyboardInterrupt: 